# WEBカメラで本のバーコードを読み取って蔵書リストを作る（Colab版）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyorin-phys/MLIntro2026/blob/main/2/isbn_colab.ipynb)

ColabのPythonはGoogleのサーバーで動いているので、`cv2.VideoCapture(0)` ではノートPCのカメラを使えない。
そこで、ブラウザ（JavaScript）でカメラの画像を撮り、それをPythonに渡してバーコードを読み取る。

* 最初に実行するとブラウザがカメラの使用許可を求めるので「許可」する
* 本の裏表紙のバーコード（上段の978から始まる方）をカメラに近づける
* 「終了」ボタンで止める

書名などは [openBD](https://openbd.jp/) から取得し、見つからなければ[国立国会図書館サーチ](https://ndlsearch.ndl.go.jp/)から取得する。自分のPCで動かす版は [isbn_scanner.py](isbn_scanner.py)

In [1]:
import time
import xml.etree.ElementTree as ET
from base64 import b64decode

import cv2
import numpy as np
import pandas as pd
import requests
from IPython.display import Javascript, Markdown, display
from google.colab.output import eval_js

ModuleNotFoundError: No module named 'google.colab'

## ISBNの処理（isbn_scanner.py と同じ）

In [ ]:
def isbn13_to_isbn10(isbn13):
    """978から始まるISBN-13をISBN-10に変換する（AmazonのURLに使う）"""
    body = isbn13[3:12]
    s = sum((10 - i) * int(d) for i, d in enumerate(body))
    check = (11 - s % 11) % 11
    return body + ('X' if check == 10 else str(check))


def amazon_url(isbn13):
    if isbn13.startswith('978'):
        return f'https://www.amazon.co.jp/dp/{isbn13_to_isbn10(isbn13)}'
    return f'https://www.amazon.co.jp/s?k={isbn13}'  # 979で始まるISBNはISBN-10がないので検索URL


def lookup_openbd(isbn13):
    """openBD から (書名, 著者, 出版社) を取得する。登録がなければ None"""
    r = requests.get('https://api.openbd.jp/v1/get', params={'isbn': isbn13}, timeout=10)
    book = r.json()[0]
    if book is None:
        return None
    s = book['summary']
    return s['title'], s['author'], s['publisher']


NS = {'dc': 'http://purl.org/dc/elements/1.1/',
      'xsi': 'http://www.w3.org/2001/XMLSchema-instance'}


def lookup_ndl(isbn13):
    """国立国会図書館サーチ から (書名, 著者, 出版社) を取得する。登録がなければ None"""
    r = requests.get('https://ndlsearch.ndl.go.jp/api/opensearch', params={'isbn': isbn13}, timeout=10)
    wanted = {isbn13}
    if isbn13.startswith('978'):
        wanted.add(isbn13_to_isbn10(isbn13))
    for item in ET.fromstring(r.content).iterfind('channel/item'):
        # 登録がないと関係のない本が返ってくることがあるので、ISBNが一致する本だけを使う
        isbns = {e.text.replace('-', '') for e in item.iterfind('dc:identifier', NS)
                 if e.text and e.get(f'{{{NS["xsi"]}}}type') in ('dcndl:ISBN', 'dcndl:ISBN13')}
        if not isbns & wanted:
            continue

        def text(tag):
            e = item.find(tag, NS)
            return e.text if e is not None and e.text else ''
        return text('dc:title'), text('dc:creator'), text('dc:publisher')
    return None


def lookup(isbn13):
    """openBD で探し、書名が得られなければ国立国会図書館サーチで探す"""
    for find in (lookup_openbd, lookup_ndl):
        try:
            book = find(isbn13)
        except Exception as e:  # 通信エラーなど
            print(f'  {find.__name__}: 取得に失敗 ({type(e).__name__})')
            continue
        if book and book[0]:
            return book
    print(f'  {isbn13}: 書名が見つからない')
    return '', '', ''

## カメラの画像を取得する（JavaScript）

`grabFrame()` を呼ぶたびに、カメラの今の画像を1枚返す。「終了」ボタンが押されたら空文字を返す。

In [ ]:
JS = '''
async function grabFrame() {
  if (!window._scan) {
    const div = document.createElement('div');
    const stop = document.createElement('button');
    stop.textContent = '終了';
    const video = document.createElement('video');
    video.style.display = 'block';
    video.width = 480;
    const stream = await navigator.mediaDevices.getUserMedia(
        {video: {width: {ideal: 1280}, height: {ideal: 720}}});
    div.appendChild(stop);
    div.appendChild(video);
    document.body.appendChild(div);
    video.srcObject = stream;
    await video.play();
    google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
    window._scan = {div: div, video: video, stream: stream, stopped: false};
    stop.onclick = () => { window._scan.stopped = true; };
  }
  const s = window._scan;
  if (s.stopped) {
    s.stream.getVideoTracks().forEach(t => t.stop());
    s.div.remove();
    window._scan = null;
    return '';
  }
  const canvas = document.createElement('canvas');
  canvas.width = s.video.videoWidth;
  canvas.height = s.video.videoHeight;
  canvas.getContext('2d').drawImage(s.video, 0, 0);
  return canvas.toDataURL('image/jpeg', 0.9);
}
'''


def grab_frame():
    """カメラの画像をnumpy配列(BGR)で返す。終了ボタンが押されたら None"""
    data = eval_js('grabFrame()')
    if not data:
        return None
    buf = np.frombuffer(b64decode(data.split(',')[1]), np.uint8)
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)

## 読み取りの実行

「終了」ボタンを押すか、`max_seconds` 秒たつと止まる。もう一度実行すると、続けて登録できる。

In [ ]:
books = []  # 登録した本のリスト（このセルを再実行すると空に戻る）

In [ ]:
def scan_books(max_seconds=300):
    display(Javascript(JS))
    detector = cv2.barcode.BarcodeDetector()
    registered = {b['ISBN'] for b in books}
    start = time.time()
    while time.time() - start < max_seconds:
        frame = grab_frame()
        if frame is None:  # 終了ボタン
            break
        ok, codes, types, points = detector.detectAndDecodeWithType(frame)
        if not ok:
            continue
        for code in codes:
            # 日本の本はバーコードが2段。上段の978/979で始まる方がISBN（下段192...は価格コード）
            if not (len(code) == 13 and code.startswith(('978', '979'))) or code in registered:
                continue
            title, author, publisher = lookup(code)
            books.append({'ISBN': code, '書名': title, '著者': author, '出版社': publisher,
                          'Amazon': amazon_url(code)})
            registered.add(code)
            print(f'登録: {code} {title} / {author}')
    else:  # 時間切れのときはカメラを止める
        eval_js('window._scan && (window._scan.stopped = true) && grabFrame()')
    print(f'{len(books)} 冊登録済み')


scan_books()

## 蔵書リストの表示と保存

In [ ]:
df = pd.DataFrame(books)
# utf-8-sig にするとExcelで開いても文字化けしない
df.to_csv('/content/books.csv', index=False, encoding='utf-8-sig')

# 書名がAmazonへのリンクになったMarkdownの表（CSVをExcelで開いてもリンクにならないため）
def cell(s):  # 表の区切りの | が書名に含まれると表が崩れるのでエスケープ
    return s.replace('|', '\\|')

lines = ['# 蔵書リスト', '', f'{len(books)} 冊', '',
         '| 書名 | 著者 | 出版社 | ISBN |', '|---|---|---|---|']
for b in books:
    title = cell(b['書名']) or '（書名不明）'
    lines.append(f"| [{title}]({b['Amazon']}) | {cell(b['著者'])} | {cell(b['出版社'])} | {b['ISBN']} |")
md = '\n'.join(lines) + '\n'
with open('/content/books.md', 'w', encoding='utf-8') as f:
    f.write(md)
display(Markdown(md))  # リンクをクリックできる表として表示

# Google Driveに保存する場合はドライブをマウントして '/content/drive/MyDrive/books.md' にする
# from google.colab import files; files.download('/content/books.md')  # PCにダウンロード